In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
import xarray as xr

In [2]:
# Define directories
Diri = '../ExtraTrack_Data/Output_Files_V7/'
Output_Diri = '../ExtraTrack_Data/NetCDF_Files_V7/'
Large_Scale_Diri = '/glade/campaign/univ/upsu0032/Hyperion_ET/'

In [3]:
# Create list of monthly file names
def Monthly_File_Names(Scenario, Years, Months, File_Names):
    for j in range(12):
        for i in range(len(Years)):
            File_Name = Scenario + '/' + Scenario + '.cam.h0.' + str(Years[i]) + '-' + str(Months[j]) + '.nc_regrid.nc'
            File_Names[j].append(File_Name)
    return (File_Names)

In [4]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)

In [5]:
# List of years for each climate scenario
Control_Years = Create_Bins(1985,2014,1)
RCP45_Years = Create_Bins(2070,2100,1)
RCP85_Years = Create_Bins(2070,2100,1)
Year_Diffs = [[84,52,20], [68,36,4], [-32,-64,-96]]
Months = ['01','02','03','04','05','06','07','08','09','10','11','12']
Months_Name = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [6]:
# Define file names for control
Control_Monthly_Files = []
for j in range(12):
    Control_Monthly_Files.append([])
Control_Monthly_Files = Monthly_File_Names('CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900', Control_Years, Months, Control_Monthly_Files)
Control_Monthly_Files = Monthly_File_Names('CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.002', Control_Years, Months, Control_Monthly_Files)
Control_Monthly_Files = Monthly_File_Names('CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.004', Control_Years, Months, Control_Monthly_Files)

In [7]:
# Define file names for RCP4.5
RCP45_Monthly_Files = []
for j in range(12):
    RCP45_Monthly_Files.append([])
RCP45_Monthly_Files = Monthly_File_Names('CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900', RCP45_Years, Months, RCP45_Monthly_Files)
RCP45_Monthly_Files = Monthly_File_Names('CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.002', RCP45_Years, Months, RCP45_Monthly_Files)
RCP45_Monthly_Files = Monthly_File_Names('CHEY.RCP45.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003', RCP45_Years, Months, RCP45_Monthly_Files)

In [8]:
# Define file names for RCP8.5
RCP85_Monthly_Files = []
for j in range(12):
    RCP85_Monthly_Files.append([])
RCP85_Monthly_Files = Monthly_File_Names('CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900', RCP85_Years, Months, RCP85_Monthly_Files)
RCP85_Monthly_Files = Monthly_File_Names('CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003', RCP85_Years, Months, RCP85_Monthly_Files)
RCP85_Monthly_Files = Monthly_File_Names('CHEY.RCP85.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.004', RCP85_Years, Months, RCP85_Monthly_Files)

In [9]:
# Create function to open monthly files
def Open_Monthly_File(Diri, File_Names, Vars, Year_Diffs):
    for i in range(len(File_Names)):
        if Vars == "All":
            Month_DS = xr.open_dataset(Diri + File_Names[i])
        else:
            Month_DS = xr.open_dataset(Diri + File_Names[i])[Vars]
        Month_DS_New = Month_DS.where((Month_DS.lon >= -100) & (Month_DS.lon <= 0) & \
        (Month_DS.lat >= 0) & (Month_DS.lat <= 60), drop=True)
# Edit time stamp to correspond to main datasets
        if i < (len(File_Names) / 3):
            Month_DS_New = Month_DS_New.assign_coords(time=pandas.to_datetime(Month_DS_New.time.values) - \
            pandas.DateOffset(years=Year_Diffs[0]))
        elif i < (len(File_Names) / 3 * 2):
            Month_DS_New = Month_DS_New.assign_coords(time=pandas.to_datetime(Month_DS_New.time.values) - \
            pandas.DateOffset(years=Year_Diffs[1]))
        else:
            Month_DS_New = Month_DS_New.assign_coords(time=pandas.to_datetime(Month_DS_New.time.values) - \
            pandas.DateOffset(years=Year_Diffs[2]))
# Combine datasets
        if i == 0:
            Month_DS_Combine = Month_DS_New
        else:
            Month_DS_Combine = xr.concat([Month_DS_Combine, Month_DS_New], dim='time')
        if i % 20 == 8:
            print (i)
# Calculate mean values across time
    Month_DS_Mean = Month_DS_Combine.mean(dim='time', keep_attrs=True)
    return (Month_DS_Mean)

In [10]:
xr.open_dataset(Large_Scale_Diri + Control_Monthly_Files[6][15])

<xarray.Dataset>
Dimensions:    (time: 1, lat: 181, lon: 361, plev: 13, nbnd: 2)
Coordinates:
  * lat        (lat) float64 -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
  * lon        (lon) float64 -180.0 -179.0 -178.0 -177.0 ... 178.0 179.0 180.0
  * plev       (plev) float64 1e+05 9.25e+04 8.5e+04 ... 1.5e+04 1e+04 5e+03
  * time       (time) datetime64[ns] 2000-08-01
Dimensions without coordinates: nbnd
Data variables: (12/74)
    CLDHGH     (time, lat, lon) float32 ...
    CLDICE     (time, plev, lat, lon) float32 ...
    CLDLIQ     (time, plev, lat, lon) float32 ...
    CLDLOW     (time, lat, lon) float32 ...
    CLDMED     (time, lat, lon) float32 ...
    CLDTOT     (time, lat, lon) float32 ...
    ...         ...
    Z3         (time, plev, lat, lon) float32 ...
    area       (lat, lon) float64 ...
    gw         (lat) float64 ...
    lat_bnds   (lat, nbnd) float64 ...
    lon_bnds   (lon, nbnd) float64 ...
    time_bnds  (time, nbnd) datetime64[ns] ...
Attributes: (12/21)
    ne:                         0
    np:                         4
    Conventions:                CF-1.0
    source:                     CAM
    title:                      Regridded version of tmp.nc
    Version:                    $Name$
    ...                         ...
    history_of_appended_files:  Sun Apr 13 12:04:21 2025: Appended file /glad...
    remap_script:               ncremap
    remap_hostname:             casper-login1
    remap_version:              5.3.1
    map_file:                   /glade/work/zarzycki/maps/hyperion/map_ne0np4...
    input_file:                 /glade/u/home/zarzycki/scratch/hyperion/CHEY....

In [11]:
# Define variables required
Vars = ['T', 'TS', 'Z3', 'PSL', 'U', 'V', 'OCNFRAC']

In [12]:
# Create dictionaries to store datasets
Control_DS_Dict = {}
RCP45_DS_Dict = {}
RCP85_DS_Dict = {}

In [13]:
# Open files for control
for i in range(12):
    print (Months_Name[i])
    Control_DS_Dict[Months_Name[i]] = Open_Monthly_File(Large_Scale_Diri, Control_Monthly_Files[i], Vars, Year_Diffs[0])

Jan
8
28
48
68
88
Feb
8
28
48
68
88
Mar
8
28
48
68
88
Apr
8
28
48
68
88
May
8
28
48
68
88
Jun
8
28
48
68
88
Jul
8
28
48
68
88
Aug
8
28
48
68
88
Sep
8
28
48
68
88
Oct
8
28
48
68
88
Nov
8
28
48
68
88
Dec
8
28
48
68
88


In [14]:
# Open files for RCP4.5
for i in range(12):
    print (Months_Name[i])
    RCP45_DS_Dict[Months_Name[i]] = Open_Monthly_File(Large_Scale_Diri, RCP45_Monthly_Files[i], Vars, Year_Diffs[1])

Jan
8
28
48
68
88
Feb
8
28
48
68
88
Mar
8
28
48
68
88
Apr
8
28
48
68
88
May
8
28
48
68
88
Jun
8
28
48
68
88
Jul
8
28
48
68
88
Aug
8
28
48
68
88
Sep
8
28
48
68
88
Oct
8
28
48
68
88
Nov
8
28
48
68
88
Dec
8
28
48
68
88


In [15]:
# Open files for RCP8.5
for i in range(12):
    print (Months_Name[i])
    RCP85_DS_Dict[Months_Name[i]] = Open_Monthly_File(Large_Scale_Diri, RCP85_Monthly_Files[i], Vars, Year_Diffs[2])

Jan
8
28
48
68
88
Feb
8
28
48
68
88
Mar
8
28
48
68
88
Apr
8
28
48
68
88
May
8
28
48
68
88
Jun
8
28
48
68
88
Jul
8
28
48
68
88
Aug
8
28
48
68
88
Sep
8
28
48
68
88
Oct
8
28
48
68
88
Nov
8
28
48
68
88
Dec
8
28
48
68
88


In [16]:
Control_DS_Dict['Sep']

<xarray.Dataset>
Dimensions:  (plev: 13, lat: 61, lon: 101)
Coordinates:
  * lat      (lat) float64 0.0 1.0 2.0 3.0 4.0 5.0 ... 56.0 57.0 58.0 59.0 60.0
  * lon      (lon) float64 -100.0 -99.0 -98.0 -97.0 -96.0 ... -3.0 -2.0 -1.0 0.0
  * plev     (plev) float64 1e+05 9.25e+04 8.5e+04 7e+04 ... 1.5e+04 1e+04 5e+03
Data variables:
    T        (plev, lat, lon) float32 293.6 293.4 293.3 ... 217.6 217.6 217.6
    TS       (lat, lon) float32 295.1 295.0 294.9 294.9 ... 285.3 285.4 285.6
    Z3       (plev, lat, lon) float32 119.9 120.0 120.0 ... 2.058e+04 2.058e+04
    PSL      (lat, lon) float32 1.014e+05 1.014e+05 ... 1.011e+05 1.012e+05
    U        (plev, lat, lon) float32 -4.003 -3.793 -3.546 ... 7.792 7.735 7.676
    V        (plev, lat, lon) float32 5.476 5.525 5.629 ... 0.7165 0.61 0.505
    OCNFRAC  (lat, lon) float32 1.0 1.0 1.0 1.0 ... 0.9908 0.9555 0.9496 0.9521
Attributes: (12/21)
    ne:                         0
    np:                         4
    Conventions:                CF-1.0
    source:                     CAM
    title:                      Regridded version of tmp.nc
    Version:                    $Name$
    ...                         ...
    history_of_appended_files:  Sun Apr 13 09:17:40 2025: Appended file /glad...
    remap_script:               ncremap
    remap_hostname:             casper-login1
    remap_version:              5.3.1
    map_file:                   /glade/work/zarzycki/maps/hyperion/map_ne0np4...
    input_file:                 /glade/u/home/zarzycki/scratch/hyperion/CHEY....

In [17]:
# Combine monthly files based on seasons (Summer, Fall, Winter-Spring)
def Combine_Monthly_Files(DS_Dict, Scenario):
    print (Scenario)
    DS_Dict['Jun-Aug'] = xr.concat([DS_Dict['Jun'], DS_Dict['Jul'], DS_Dict['Aug']], dim='time').mean(dim='time')
    DS_Dict['Sep-Nov'] = xr.concat([DS_Dict['Sep'], DS_Dict['Oct'], DS_Dict['Nov']], dim='time').mean(dim='time')
    DS_Dict['Dec-May'] = xr.concat([DS_Dict['Dec'], DS_Dict['Jan'], DS_Dict['Feb'], \
    DS_Dict['Mar'], DS_Dict['Apr'], DS_Dict['May']], dim='time').mean(dim='time')
    DS_Dict['Annual'] = xr.concat([DS_Dict['Dec-May'], DS_Dict['Jun-Aug'], DS_Dict['Sep-Nov']], dim='time').mean(dim='time')
    return (DS_Dict)

In [18]:
# Combine monthly files based on seasons
Control_DS_Dict = Combine_Monthly_Files(Control_DS_Dict, 'Control')
RCP45_DS_Dict = Combine_Monthly_Files(RCP45_DS_Dict, 'RCP4.5')
RCP85_DS_Dict = Combine_Monthly_Files(RCP85_DS_Dict, 'RCP8.5')

Control
RCP4.5
RCP8.5


In [19]:
# Create function to calculate 850-200hPa vertical wind shear
def Vertical_Wind_Shear(DS):
# Get U and V at 200hPa and 850hPa
    U200 = DS['U'].sel(plev=20000)
    U850 = DS['U'].sel(plev=85000)
    V200 = DS['V'].sel(plev=20000)
    V850 = DS['V'].sel(plev=85000)
# Calculate VWS and place into dataset
    DS['VWS_850_200'] = numpy.sqrt((U200 - U850)**2 + (V200 - V850) ** 2)
# Add metadata
    DS['VWS_850_200'].attrs = {'units': 'm s^-1', 'long_name': 'Vertical wind shear magnitude between 200 and 850hPa'}
# Output dataset
    return (DS)

In [20]:
# Create function to calculate Eady Growth Rate
def Eady_Growth_Rate(DS):
# Define constants and variables
    P0 = 100000
    Rd = 287
    cp = 1005
    Omega = 7.2921 * 10**-5
    Temp = DS['T']
    Plev = DS['plev']
    Lat = DS['lat']
    Z = DS['Z3']
    U = DS['U']
    V = DS['V']
# Calculate potential temperature
    Theta = Temp * (P0 / Plev) ** (Rd/cp)
# Calculate Coriolis Parameter
    f = 2 * Omega * numpy.sin(Lat / 180 * numpy.pi)
# Calculate Brunt Vaisala Frequency
    dTheta = Theta.diff(dim='plev')
    dZ = Z.diff(dim='plev')
    Theta_Mid = 0.5 * (Theta.isel(plev=slice(0, -1)).values + Theta.isel(plev=slice(1, None)).values)
    Plev_Mid = 0.5 * (Plev.isel(plev=slice(0, -1)).values + Plev.isel(plev=slice(1, None)).values)
    Theta_Mid = xr.DataArray(Theta_Mid, coords=dTheta.coords, dims=dTheta.dims)
    N2 = (dTheta / dZ) * 9.81 / Theta_Mid
    N2_Safe = N2.where(N2 > 0)
    N = numpy.sqrt(N2_Safe)
# Calculate vertical gradient of geostrophic wind
    dU = U.diff(dim='plev')
    dV = V.diff(dim='plev')
    dU_dZ = numpy.sqrt((dU / dZ) ** 2 + (dV / dZ) ** 2)
# Calculate Eady Growth Rate
    Sigma = 0.31 * numpy.abs(dU_dZ) * f / N
# Convert to day^-1 and place into dataset
    DS['EGR'] = Sigma * 24 * 60 * 60
# Add metadata
    DS['EGR'].attrs = {'units': 'day^-1', 'long_name': 'Eady Growth Rate'}
# Output dataset
    return (DS)

In [21]:
# Add additional variables to datasets
def Add_Variables(DS_Dict):
    for Month in DS_Dict:
        DS_Dict[Month] = Vertical_Wind_Shear(DS_Dict[Month])
        DS_Dict[Month] = Eady_Growth_Rate(DS_Dict[Month])
    return (DS_Dict)

In [22]:
# Add vertical wind shear and Eady Growth Rate to datasets
Control_DS_Dict = Add_Variables(Control_DS_Dict)
RCP45_DS_Dict = Add_Variables(RCP45_DS_Dict)
RCP85_DS_Dict = Add_Variables(RCP85_DS_Dict)

In [23]:
RCP85_DS_Dict['Sep']

<xarray.Dataset>
Dimensions:      (lat: 61, lon: 101, plev: 13)
Coordinates:
  * lat          (lat) float64 0.0 1.0 2.0 3.0 4.0 ... 56.0 57.0 58.0 59.0 60.0
  * lon          (lon) float64 -100.0 -99.0 -98.0 -97.0 ... -3.0 -2.0 -1.0 0.0
  * plev         (plev) float64 1e+05 9.25e+04 8.5e+04 ... 1.5e+04 1e+04 5e+03
Data variables:
    T            (plev, lat, lon) float32 298.2 298.1 298.0 ... 217.4 217.4
    TS           (lat, lon) float32 299.8 299.7 299.7 ... 287.6 287.8 288.1
    Z3           (plev, lat, lon) float32 120.0 120.0 ... 2.084e+04 2.084e+04
    PSL          (lat, lon) float32 1.013e+05 1.013e+05 ... 1.014e+05 1.014e+05
    U            (plev, lat, lon) float32 -3.516 -3.313 -3.094 ... 10.67 10.58
    V            (plev, lat, lon) float32 5.993 6.002 6.052 ... 1.74 1.606 1.474
    OCNFRAC      (lat, lon) float32 1.0 1.0 1.0 1.0 ... 0.9555 0.9496 0.9521
    VWS_850_200  (lat, lon) float32 8.498 8.299 8.046 ... 13.23 13.38 13.55
    EGR          (plev, lat, lon) float64 nan nan nan ... 0.1625 0.1609 0.1594
Attributes: (12/21)
    ne:                         0
    np:                         4
    Conventions:                CF-1.0
    source:                     CAM
    case:                       b40_1850_2d_r07c5cn_160jp
    title:                      Regridded version of tmp.nc
    ...                         ...
    history_of_appended_files:  Sat Apr 12 19:03:23 2025: Appended file /glad...
    remap_script:               ncremap
    remap_hostname:             casper-login2
    remap_version:              5.3.1
    map_file:                   /glade/work/zarzycki/maps/hyperion/map_ne0np4...
    input_file:                 /glade/u/home/zarzycki/scratch/hyperion/CHEY....

In [31]:
RCP45_DS_Dict['Sep']['EGR'].sel(plev=50000)

<xarray.DataArray 'EGR' (lat: 61, lon: 101)>
array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.01339314, 0.01391547, 0.01438514, ..., 0.00238091, 0.00267832,
        0.00298559],
       [0.0255618 , 0.02660492, 0.0275487 , ..., 0.00513524, 0.0056125 ,
        0.0060623 ],
       ...,
       [0.48537535, 0.48865788, 0.49122132, ..., 0.39429072, 0.39886751,
        0.40068512],
       [0.46893962, 0.47168437, 0.47325646, ..., 0.39023407, 0.39470791,
        0.39730726],
       [0.45215665, 0.45328954, 0.45392484, ..., 0.38048746, 0.38430004,
        0.38746072]])
Coordinates:
  * lat      (lat) float64 0.0 1.0 2.0 3.0 4.0 5.0 ... 56.0 57.0 58.0 59.0 60.0
  * lon      (lon) float64 -100.0 -99.0 -98.0 -97.0 -96.0 ... -3.0 -2.0 -1.0 0.0
    plev     float64 5e+04
Attributes:
    units:      day^-1
    long_name:  Eady Growth Rate

In [24]:
# Output dataset to netCDF file
def Output_DS(Output_Diri, DS, Scenario, Month):
    File_Name = str(Scenario+'_Output_'+Month+'.nc')
    DS.to_netcdf(Output_Diri+File_Name)

In [25]:
# Output control files for each month
for Month in Control_DS_Dict:
    print (Month)
    Output_DS(Output_Diri, Control_DS_Dict[Month], 'Control', Month)

Jan
Feb
Mar
Apr
May
Jun
Jul
Aug
Sep
Oct
Nov
Dec
Jun-Aug
Sep-Nov
Dec-May
Annual


In [26]:
# Output RCP4.5 files for each month
for Month in RCP45_DS_Dict:
    print (Month)
    Output_DS(Output_Diri, RCP45_DS_Dict[Month], 'RCP45', Month)

Jan
Feb
Mar
Apr
May
Jun
Jul
Aug
Sep
Oct
Nov
Dec
Jun-Aug
Sep-Nov
Dec-May
Annual


In [27]:
# Output RCP8.5 files for each month
for Month in RCP85_DS_Dict:
    print (Month)
    Output_DS(Output_Diri, RCP85_DS_Dict[Month], 'RCP85', Month)

Jan
Feb
Mar
Apr
May
Jun
Jul
Aug
Sep
Oct
Nov
Dec
Jun-Aug
Sep-Nov
Dec-May
Annual
